In [2]:
import torch
import numpy as np
import scipy
import matplotlib.pyplot as plt

from sbi import utils as utils
from sbi import analysis as analysis
from sbi.inference.base import infer

/Users/rutongpei/opt/anaconda3/envs/sbi_env/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
from  subhalo_impact import chi_eval
from rotation_matrix import obs_from_pos6d

In [6]:
# # log M_sat, r, vz
# num_dim = 3

# # need to change to more accurate distributions (range, mean, etc.)
# prior_min = [0.00001, -50, 0]
# prior_max = [10, 0, 30]


num_dim = 1
prior_min = [-5]
prior_max = [1]

prior_uniform = utils.BoxUniform(low = torch.as_tensor(prior_min), high=torch.as_tensor(prior_max))

prior_normal = torch.distributions.MultivariateNormal(5 * torch.ones(num_dim), 1 * torch.eye(num_dim))

In [23]:
def simulator (params, scatter = 0.1, bins = 100):
    params = np.array(params) # convert tensor to ny ndarray
    M_sat = 10**params[0]
    # vz = params[1]
    # r = params[2]

    vz = -20
    r = 0.2
    phi = 250 # angle around stream in dec
    vphi = 35  # velocity around stream in km/s
    t_a = 0.2 # time since interaction in Gyr
    phi_a = -4 # interaction point along stream in deg, phi=0 is progenitor location (try -20 to 10)
    rs_sat = 1.05 * (M_sat*10*10)**0.5 # scale radius of subhalo in kpc, can be adjusted along with M_sat using equation 15 in erkal et al. 2015
    pid=36 # index included in saved filenames
    tmax=4 # how long stream disrupts in Gyr

    chi_eval(r,phi,vphi,vz,M_sat,tmax,t_a,phi_a,rs_sat,pid)
    data = np.genfromtxt(f'final_stream/final_stream_{pid}.txt')
    R_phi12_radec = np.array([[0.83697865, 0.29481904, -0.4610298], 
                          [0.51616778, -0.70514011, 0.4861566], 
                          [0.18176238, 0.64487142, 0.74236331]])
    phi1,phi2,dist,pm1,pm2,vr = obs_from_pos6d(data[:,:3],data[:,3:6],R_phi12_radec)

    '''to include in the output:
        phi1 edges
        fraction = count / len(phi1)
        phi2 median
        phi2 std
        vr median
        vr std
    '''

    phi2_median, phi1_edges, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='median', bins = bins)
    phi2_std, _, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='std', bins = bins)
    count, _, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='count', bins = bins)
    vr_median, _, _ = scipy.stats.binned_statistic(phi1, vr, statistic='median', bins = bins)
    vr_std, _, _ = scipy.stats.binned_statistic(phi1, vr, statistic='std', bins = bins)

    output = np.concatenate((phi1_edges, phi2_median, phi2_std, count/len(phi1), vr_median, vr_std))
    print(np.sum(np.isnan(output)))
    output = torch.from_numpy(output)
    print(output + torch.randn_like(output) * scatter)
    return output + torch.randn_like(output) * scatter

In [24]:
simulator([7.5])

Setup pid: 36 with tmax=4.000000, t_approach=0.200000 
Sanity check : Initializing potential
tmax: 3.886260
final_t: 3.886260
diff: 0.000000
Running orbit with r=0.2,phi=250,vphi=35,vz=-20,tapproach=0.2,phiapproach=-4,pid=36
Orbit pid: 36 with x=-13.052411, y=-33.619492, z=-23.715755, vx=-59.608256, vy=25.286811, vz=-54.697573, tmax=0.204642,  
Sanity check : Initializing potential
Impact with scale radius:  28.755434269021222
Impact pid: 36 with M_sat=7.500000, rs_sat=28.755434,  
Sanity check : Initializing potential
16
tensor([-9.6701e+00, -9.7283e+00, -9.2554e+00, -9.2507e+00, -8.9927e+00,
        -8.6575e+00, -8.2470e+00, -8.0372e+00, -7.9718e+00, -7.6545e+00,
        -7.5167e+00, -7.2162e+00, -6.8363e+00, -6.8254e+00, -6.5904e+00,
        -6.3765e+00, -5.9827e+00, -5.9234e+00, -5.8119e+00, -5.3797e+00,
        -5.1495e+00, -5.2008e+00, -4.7567e+00, -4.2238e+00, -4.2799e+00,
        -4.0310e+00, -3.6286e+00, -3.4982e+00, -3.4663e+00, -3.2008e+00,
        -2.8081e+00, -2.5482e+00, 

tensor([-9.6308e+00, -9.4143e+00, -9.3115e+00, -9.1022e+00, -8.9459e+00,
        -8.4701e+00, -8.3837e+00, -8.1920e+00, -7.8927e+00, -7.5425e+00,
        -7.5516e+00, -7.2575e+00, -7.1252e+00, -6.8340e+00, -6.5155e+00,
        -6.3891e+00, -6.1429e+00, -5.8071e+00, -5.5879e+00, -5.5265e+00,
        -5.1727e+00, -5.0064e+00, -4.6148e+00, -4.6774e+00, -4.4890e+00,
        -4.0878e+00, -3.7891e+00, -3.5907e+00, -3.1781e+00, -3.0805e+00,
        -2.8134e+00, -2.5551e+00, -2.2096e+00, -2.3127e+00, -1.9231e+00,
        -1.6091e+00, -1.3162e+00, -1.2768e+00, -1.1757e+00, -8.8054e-01,
        -6.6127e-01, -4.8832e-01,  1.2940e-02,  4.0289e-02,  2.2778e-01,
         6.0947e-01,  7.0165e-01,  9.5824e-01,  1.1171e+00,  1.4023e+00,
         1.6668e+00,  1.5831e+00,  2.1571e+00,  2.4616e+00,  2.5937e+00,
         3.0088e+00,  2.9166e+00,  3.1547e+00,  3.6311e+00,  3.7105e+00,
         3.8858e+00,  4.2373e+00,  4.4419e+00,  4.7099e+00,  4.7415e+00,
         5.0403e+00,  5.4235e+00,  5.5045e+00,  5.6

In [19]:
posterior = infer(simulator, prior_uniform, "SNPE", num_simulations=2)

Setup pid: 36 with tmax=4.000000, t_approach=0.200000 
Sanity check : Initializing potential
tmax: 3.886260
final_t: 3.886260
diff: 0.000000
Running orbit with r=0.2,phi=250,vphi=35,vz=-20,tapproach=0.2,phiapproach=-4,pid=36
Orbit pid: 36 with x=-13.052411, y=-33.619492, z=-23.715755, vx=-59.608256, vy=25.286811, vz=-54.697573, tmax=0.204642,  
Sanity check : Initializing potential
Impact with scale radius:  [15.50217]
Impact pid: 36 with M_sat=0.000000, rs_sat=0.000000,  
Sanity check : Initializing potential
tensor([-3.0547e+01, -2.9222e+01, -2.7953e+01, -2.6613e+01, -2.5461e+01,
        -2.4245e+01, -2.3110e+01, -2.1687e+01, -2.0303e+01, -1.9103e+01,
        -1.7791e+01, -1.6583e+01, -1.5605e+01, -1.4222e+01, -1.2647e+01,
        -1.1750e+01, -1.0233e+01, -9.0750e+00, -7.8828e+00, -6.6243e+00,
        -5.3641e+00, -4.0202e+00, -2.8137e+00, -1.6027e+00, -2.1339e-01,
         1.0587e+00,  2.3069e+00,  3.3590e+00,  5.0232e+00,  5.8892e+00,
         7.1927e+00,  8.5821e+00,  9.7872e+00,

Running 2 simulations.:   0%|          | 0/2 [00:00<?, ?it/s]

Setup pid: 36 with tmax=4.000000, t_approach=0.200000 
Sanity check : Initializing potential
tmax: 3.886260
final_t: 3.886260
diff: 0.000000
Running orbit with r=0.2,phi=250,vphi=35,vz=-20,tapproach=0.2,phiapproach=-4,pid=36
Orbit pid: 36 with x=-13.052411, y=-33.619492, z=-23.715755, vx=-59.608256, vy=25.286811, vz=-54.697573, tmax=0.204642,  
Sanity check : Initializing potential
Impact with scale radius:  21.470793347796796
Impact pid: 36 with M_sat=4.181360, rs_sat=21.470793,  
Sanity check : Initializing potential


Running 2 simulations.:  50%|█████     | 1/2 [00:45<00:45, 45.28s/it]

tensor([-6.6369e+00, -6.5225e+00, -6.3695e+00, -6.1719e+00, -6.0126e+00,
        -5.8688e+00, -5.5842e+00, -5.4074e+00, -5.3860e+00, -5.2128e+00,
        -5.0645e+00, -4.8503e+00, -4.8754e+00, -4.5640e+00, -4.3819e+00,
        -4.1811e+00, -4.0518e+00, -3.9401e+00, -3.7655e+00, -3.5113e+00,
        -3.4263e+00, -3.7295e+00, -3.3859e+00, -3.2341e+00, -3.0135e+00,
        -2.6860e+00, -2.4808e+00, -2.4471e+00, -2.4870e+00, -1.9643e+00,
        -1.8593e+00, -1.7241e+00, -1.5940e+00, -1.4943e+00, -1.5001e+00,
        -1.2530e+00, -1.2468e+00, -9.0483e-01, -6.9444e-01, -6.1269e-01,
        -2.6602e-01, -4.1031e-02,  9.7163e-02,  6.3630e-02,  3.3382e-01,
         4.3593e-01,  4.0213e-01,  7.3597e-01,  8.4054e-01,  8.8563e-01,
         1.2426e+00,  1.3397e+00,  1.3433e+00,  1.7789e+00,  1.8544e+00,
         1.9031e+00,  1.9604e+00,  2.2066e+00,  2.2728e+00,  2.5630e+00,
         2.6201e+00,  2.8091e+00,  3.1007e+00,  3.2163e+00,  3.2953e+00,
         3.3264e+00,  3.4957e+00,  3.6815e+00,  3.7

Running 2 simulations.: 100%|██████████| 2/2 [01:37<00:00, 48.80s/it]

tensor([-9.8672e+00, -9.5893e+00, -9.2216e+00, -9.0316e+00, -8.9431e+00,
        -8.5031e+00, -8.4461e+00, -8.3213e+00, -8.0112e+00, -7.7475e+00,
        -7.3908e+00, -7.0610e+00, -7.0437e+00, -6.7881e+00, -6.5197e+00,
        -6.2218e+00, -6.1398e+00, -5.6716e+00, -5.4845e+00, -5.3207e+00,
        -5.0941e+00, -4.9821e+00, -4.8834e+00, -4.5286e+00, -4.2857e+00,
        -4.0897e+00, -3.6943e+00, -3.4152e+00, -3.2768e+00, -2.9738e+00,
        -2.7276e+00, -2.5186e+00, -2.2991e+00, -2.3020e+00, -1.9470e+00,
        -1.6487e+00, -1.7191e+00, -1.1807e+00, -6.8829e-01, -7.6856e-01,
        -3.9731e-01, -2.9878e-01, -1.7787e-02,  3.1061e-01,  4.6829e-01,
         7.8775e-01,  9.3623e-01,  1.1315e+00,  1.3582e+00,  1.5842e+00,
         1.7749e+00,  2.1793e+00,  2.4530e+00,  2.6056e+00,  2.6908e+00,
         3.0193e+00,  3.2843e+00,  3.5310e+00,  3.7579e+00,  3.8746e+00,
         4.3428e+00,  4.4400e+00,  4.4730e+00,  4.9064e+00,  5.1235e+00,
         5.4015e+00,  5.4946e+00,  5.9309e+00,  5.9

ValueError: batch_size should be a positive integer value, but got batch_size=0

In [ ]:
# params = [0.01,-20,0.2]
params = [3]
observation1 = simulator(torch.as_tensor(params))

In [ ]:
posterior_samples_1 = posterior.sample((1000,), x=observation1)

In [ ]:
_ = analysis.pairplot(
    posterior_samples_1, limits = [[0.00001,10], [-100,0],[0,30]],
    ticks = [[0.00001,10], [-100,0],[0,30]],
    figsize = (5, 5),
    points = np.array(params),
    points_offdiag = {"markersize": 6},
    points_colors = "r",
    labels = ["M_sat","vz","r"]
)

In [30]:
bins = 100
pid = 36
data = np.genfromtxt(f'final_stream/final_stream_{pid}.txt')
R_phi12_radec = np.array([[0.83697865, 0.29481904, -0.4610298], 
                          [0.51616778, -0.70514011, 0.4861566], 
                          [0.18176238, 0.64487142, 0.74236331]])
phi1,phi2,dist,pm1,pm2,vr = obs_from_pos6d(data[:,:3],data[:,3:6],R_phi12_radec)

phi2_median, phi1_edges, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='median', bins = bins)
phi2_std, _, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='std', bins = bins)
count, _, _ = scipy.stats.binned_statistic(phi1, phi2, statistic='count', bins = bins)
vr_median, _, _ = scipy.stats.binned_statistic(phi1, vr, statistic='median', bins = bins)
vr_std, _, _ = scipy.stats.binned_statistic(phi1, vr, statistic='std', bins = bins)
# (phi1_edges, phi2_median, phi2_std, count/len(phi1), vr_median, vr_std)

# print(np.sum(np.isnan(phi1_edges)))
# print(np.sum(np.isnan(phi2_median)))
# print(np.sum(np.isnan(phi2_std)))
print(phi1_edges[:-1][count==0])
print(min(phi1), max(phi1))
# print(np.sum(np.isnan(vr_median)))
# print(np.sum(np.isnan(vr_std)))

print(np.sum(np.isnan(vr)))
print(np.sum(np.isnan(phi2)))


[-9.51236029 -9.05556446 10.35825821 10.58665613 11.27184987 11.7286457
 12.41383944 12.64223735]
-9.740758202970355 13.09903317787274
0
0
